In [1]:
from libero.libero import benchmark
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from openpi_client import image_tools

import pathlib
import numpy as np
import math
import os

CAMERA_NAMES = ["agentview", "birdview", "robot0_eye_in_hand", "sideview", "canonical_frontview"]

def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution, "camera_names": CAMERA_NAMES}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description

def _get_empty_env(task, env):
    import robosuite
    dataset_file = os.path.join(get_libero_path("datasets"), f"{task.problem_folder}/{task.name}_demo.hdf5")
    import h5py
    import json
    f = h5py.File(dataset_file, "r")
    env_meta = json.loads(f["data"].attrs["env_args"])
    f.close()
    empty_env_kwargs = env_meta['env_kwargs'].copy()
    empty_env_kwargs['env_name'] = "SingleArmEmptyEnv"
    empty_env_kwargs['hard_reset'] = False
    empty_env_kwargs['ignore_done'] = True
    empty_env_kwargs['has_offscreen_renderer'] = False
    empty_env_kwargs['has_renderer'] = False
    empty_env_kwargs['use_camera_obs'] = False
    empty_env_kwargs['camera_names'] = CAMERA_NAMES
    empty_env_kwargs['camera_heights'] = LIBERO_ENV_RESOLUTION
    empty_env_kwargs['camera_widths'] = LIBERO_ENV_RESOLUTION
    empty_env_kwargs['robots'] = [type(robot.robot_model).__name__ for robot in env.robots]
    empty_env = robosuite.make(**empty_env_kwargs)
    empty_env.copy_env_model(env)
    return empty_env


task_suite_name = "libero_object"
task_id = 1
seed = 0
from vlm_agent import VLMAgent
from vlm_utils import *
agent = VLMAgent(task_suite_name, task_id)

LIBERO_ENV_RESOLUTION = 224
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[task_suite_name]()
task = task_suite.get_task(task_id)
initial_states = task_suite.get_task_init_states(task_id)
env, task_description = _get_libero_env(task, LIBERO_ENV_RESOLUTION, seed)
empty_env = _get_empty_env(task, env)
print(f"Task description: {task_description}")

from dp_utils import embed_lang
subtask_description = task_description.replace(" up", "")
subtask_embedding = embed_lang(subtask_description)

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/scripts/setup_macros.py (macros.py:55)
/n/holylabs/ydu_lab/Lab/zhangxiangcheng/miniconda3/envs/libero_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


Number of CUDA devices available: 2


Loading weights: 100%|█████████████████████| 1468/1468 [00:00<00:00, 13423.94it/s]


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card4: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card3: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card2: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied



Task description: pick up the cream cheese and place it in the basket


Loading weights: 100%|███████████████████████| 197/197 [00:00<00:00, 14206.73it/s]
CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.

ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robomimic/robomimic/scripts/setup_macros.py
)
Loaded language embed from cache.


In [2]:
env.reset()
for _ in range(10):
    obs, reward, done, info = env.step(LIBERO_DUMMY_ACTION)
from PIL import Image
Image.fromarray(obs["agentview_image"][::-1]).save("agentview_image.png")
Image.fromarray(obs["birdview_image"][::-1]).save("birdview_image.png")
Image.fromarray(obs['sideview_image'][::-1]).save("sideview_image.png")

In [2]:
from wm_client.client import WMClient
from wm_client.wm_env import WMEnv
host = "0.0.0.0"
port = 7880
wm_client = WMClient(host, port)

Connecting to ws://0.0.0.0:7880...
Connected to ws://0.0.0.0:7880


In [3]:
env.reset()
wm_env = WMEnv(env, empty_env, wm_client)
# wm_env = env
num_steps = 0
replay_images = []
obs = wm_env.reset()
# obs = env.set_init_state(initial_states[seed])
for t in range(10):
    obs, reward, done, info = wm_env.step(LIBERO_DUMMY_ACTION)
agent.start_episode(obs)

/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]


In [4]:
from dp_utils import load_checkpoint
import robosuite.utils.transform_utils as T
import torch
device_count = torch.cuda.device_count()
print(f"Number of CUDA devices available: {device_count}")
device = f"cuda:{device_count-1}" 
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.20/05.27.36_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0075-test_mean_score=1.000.ckpt"
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.02.27/20.14.00_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0022-test_mean_score=1.000.ckpt"
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.21/22.48.59_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0035-test_mean_score=0.900.ckpt"
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.20/06.20.27_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0540-test_mean_score=0.800.ckpt"
checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.14/08.49.11_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0460-test_mean_score=0.100.ckpt"
policy, cfg = load_checkpoint(checkpoint_path)
policy = policy.to(device)
def to_torch(image):
    image = image_tools.resize_with_pad(image, 128, 128)
    return np.moveaxis(image[::-1], -1, 0) / 255.0
def policy_fn(obs):
    np_obs_dict = dict(obs)
    if "lang_embed" in cfg.shape_meta.obs:
        np_obs_dict["lang_embed"] = subtask_embedding
    obs_keys = cfg.shape_meta.obs.keys()
    np_obs_dict = {k: np_obs_dict[k] for k in obs_keys}
    np_obs_dict = {k: to_torch(v) if "image" in k else v for k, v in np_obs_dict.items()}
    obs_dict = {k: torch.from_numpy(v).to(device).unsqueeze(0).unsqueeze(0) for k, v in np_obs_dict.items()}
    with torch.no_grad():
        action_dict = policy.predict_action(obs_dict)
    np_action_dict = {k: v.cpu().numpy() for k, v in action_dict.items()}
    action = np_action_dict['action_pred'][0]
    return action

# idm_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.17/08.56.15_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0110-val_loss=0.019.ckpt"
idm_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.12/01.36.19_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0020-val_loss=0.025.ckpt"
idm, idm_cfg = load_checkpoint(idm_checkpoint_path)
idm = idm.to(device)
def idm_fn(obs, target_pos, target_quat=None):
    np_obs_dict = dict(obs)
    obs_keys = idm_cfg.shape_meta.obs.keys()
    np_obs_dict = {k: np_obs_dict[k] for k in obs_keys}
    delta_obs_dict = {"robot0_eef_pos": target_pos - obs['robot0_eef_pos']}
    if target_quat is not None:
        delta_obs_dict['robot0_eef_quat'] = T.quat_distance(target_quat, obs['robot0_eef_quat'])
    obs_dict = {k: torch.from_numpy(v).to(device).unsqueeze(0) for k, v in np_obs_dict.items()}
    delta_obs_dict = {k: torch.from_numpy(v).to(device).unsqueeze(0) for k, v in delta_obs_dict.items()}
    with torch.no_grad():
        action_dict = idm.predict_action(obs_dict, delta_obs_dict)
    np_pred_action = action_dict['action_pred'].cpu().numpy()[0]
    return np_pred_action

# idm_2_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.17/23.50.14_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0060-val_loss=0.020.ckpt"
idm_2_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.11/21.52.52_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0040-val_loss=0.026.ckpt"
idm_2, idm_2_cfg = load_checkpoint(idm_2_checkpoint_path)
idm_2 = idm_2.to(device)
def idm_fn_2(obs, target_pos, target_quat=None):
    np_obs_dict = dict(obs)
    obs_keys = idm_2_cfg.shape_meta.obs.keys()
    np_obs_dict = {k: np_obs_dict[k] for k in obs_keys}
    delta_obs_dict = {"robot0_eef_pos": target_pos - obs['robot0_eef_pos']}
    if target_quat is not None:
        delta_obs_dict['robot0_eef_quat'] = T.quat_distance(target_quat, obs['robot0_eef_quat'])
    obs_dict = {k: torch.from_numpy(v).to(device).unsqueeze(0) for k, v in np_obs_dict.items()}
    delta_obs_dict = {k: torch.from_numpy(v).to(device).unsqueeze(0) for k, v in delta_obs_dict.items()}
    with torch.no_grad():
        action_dict = idm_2.predict_action(obs_dict, delta_obs_dict)
    np_pred_action = action_dict['action_pred'].cpu().numpy()[0]
    return np_pred_action

Number of CUDA devices available: 2



============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos', 'lang_embed']
using obs modality: rgb with keys: ['agentview_image', 'robot0_eye_in_hand_image']
using obs modality: depth with keys: []
using obs modality: scan with keys: []


In [14]:
import tqdm
pbar = tqdm.tqdm(total=500, desc="Executing policy")
while not done and num_steps < 20:
    action_chunk = policy_fn(obs)[:10]
    for action in (action_chunk):
        obs, reward, done, info = wm_env.step(action)
        replay_images.append(obs["agentview_image"][::-1])
    pbar.update(10)
    num_steps += 10
        
    

Executing policy:   4%|█                         | 20/500 [02:07<51:09,  6.39s/it]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]


In [5]:
target_object_position = agent.identify_target_object()
# target_object_position = agent.get_action_proposal()  
plot_coordinates_on_image(obs, target_object_position)
target_point = generate_3d_point(target_object_position, empty_env.get_camera_info())
target_point[2] += 0.08
action_chunk = update_gripper_action(idm_fn(obs, target_point), -1)
for action in action_chunk:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs["agentview_image"][::-1])
for _ in range(2):
    action_chunk = policy_fn(obs)[:10]
    for action in (action_chunk):
        obs, reward, done, info = wm_env.step(action)
        replay_images.append(obs["agentview_image"][::-1])

Found 1 objects
Found 1 objects
Found 1 objects


/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]


In [40]:
agent.start_mpc(obs)
agent_actions = agent.get_action_proposal()
for action_dict in agent_actions:
    if action_dict["action"] == "MOVE":
        plot_coordinates_on_image(obs, action_dict['parameters'])
        target_point = generate_3d_point(action_dict['parameters'], empty_env.get_camera_info())
        target_quat = None
        action_chunk = idm_fn(obs, target_point)
        action_chunk = update_gripper_action(action_chunk, 1)
        break 

Time taken for OpenAI API call: 10.48 seconds
Token used:  ResponseUsage(input_tokens=1098, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=413, output_tokens_details=OutputTokensDetails(reasoning_tokens=250), total_tokens=1511)
API response: [
  {
    "action": "MOVE",
    "parameters": {
      "frontview": {"x": 825, "y": 500},
      "topview": {"x": 810, "y": 495},
      "sideview": {"x": 300, "y": 610}
    }
  },
  {
    "action": "MOVE",
    "parameters": {
      "frontview": {"x": 825, "y": 585},
      "topview": {"x": 810, "y": 495},
      "sideview": {"x": 300, "y": 700}
    }
  },
  {
    "action": "RELEASE",
    "parameters": {}
  }
]


In [6]:
agent.start_mpc(obs)
place_actions = agent.place_proposal()
plot_coordinates_on_image(obs, place_actions)
target_point = generate_3d_point(place_actions, empty_env.get_camera_info())
target_quat = None
action_chunk = idm_fn(obs, target_point)
action_chunk = update_gripper_action(action_chunk, 1)


Time taken for OpenAI API call: 8.78 seconds
Token used:  ResponseUsage(input_tokens=433, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=257, output_tokens_details=OutputTokensDetails(reasoning_tokens=194), total_tokens=690)
API response for placement proposal: ```json
{
  "frontview": {"x": 795, "y": 455},
  "topview": {"x": 812, "y": 549},
  "sideview": {"x": 272, "y": 688}
}
```


In [7]:
import imageio
with wm_env.simulation():
    pred_obs = wm_env.simulate(action_chunk)
    wm_agent_obs = pred_obs['future_obs']
imageio.mimwrite('test_dp_output_wm_1.mp4', pred_obs['WMPredictionOutput'].full_video, fps=20)

In [ ]:
endpoint_response = agent.optimize_endpoint(wm_agent_obs)

In [ ]:
target_point += optimize_endpoint(endpoint_response, scale=0.1)

In [ ]:
height_response = agent.optimize_height_sideview(wm_agent_obs)

In [ ]:
target_point += np.array([0, 0, height_response]) * 0.1

In [ ]:
candidate_points = generate_candidates(target_point, scale=0.05)

In [8]:
trajectory_response = agent.optimize_trajectory(wm_agent_obs)

Time taken for OpenAI API call: 40.93 seconds
Token used:  ResponseUsage(input_tokens=1794, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=348, output_tokens_details=OutputTokensDetails(reasoning_tokens=240), total_tokens=2142)
API response for trajectory optimization: The trajectory is carrying the cream cheese toward the basket on the right. The main risk is near the basket approach and placement: the object/gripper appears low and close to the basket rim, especially the near-left side of the basket opening. A small upward adjustment will improve rim clearance, and a slight rightward adjustment will better center the object over the basket.

```json
{
  "delta_x": 0,
  "delta_y": 1,
  "delta_z": 1
}
```


In [9]:
adjustment = optimize_trajectory(trajectory_response, scale=0.1)
target_point = target_point + adjustment
action_chunk = idm_fn(obs, target_point)
action_chunk = update_gripper_action(action_chunk, 1)

In [ ]:
for action in action_chunk[:20]:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs["agentview_image"][::-1])
    num_steps += 1
action_chunk = idm_fn(obs, target_point)


/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]


In [10]:
import imageio
candidate_obs = []
candidate_actions = []
candidate_points = [target_point, target_point + np.array([-0.05, 0, 0]), target_point + np.array([0.05, 0, 0])]
for i, candidate_point in enumerate(candidate_points):
    with wm_env.simulation():
        candidate_action = update_gripper_action(idm_fn(obs, candidate_point), 1)
        candidate_actions.append(candidate_action)
        next_obs = wm_env.simulate(candidate_action)
        # next_action_chunk = policy_fn(next_obs['future_obs'][-1])[:20]
        # next_obs = wm_env.simulate(next_action_chunk)
        # next_action_chunk = policy_fn(next_obs['future_obs'][-1])[:20]
        # next_obs = wm_env.simulate(next_action_chunk)
    imageio.mimwrite(f'test_dp_output_candidate{i}.mp4', next_obs['WMPredictionOutput'].full_video, fps=20)
    candidate_obs.append(next_obs['future_obs'])

In [11]:
# frontview_ranking = agent.rank_images_frontview(candidate_obs)
# wristview_ranking = agent.rank_images_wristview(candidate_obs)
sideview_ranking = agent.rank_placement_sideview(candidate_obs)

Time taken for OpenAI API call: 12.91 seconds
Token used:  ResponseUsage(input_tokens=588, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=116, output_tokens_details=OutputTokensDetails(reasoning_tokens=97), total_tokens=704)
API response for sideview placement ranking: ```json
[1, 0, 2]
```


In [12]:
action_chunk = candidate_actions[sideview_ranking[0]]

In [13]:
for action in action_chunk:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs["agentview_image"][::-1])

/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]


In [15]:
for _ in range(20):
    obs, reward, done, info = wm_env.step(LIBERO_DUMMY_ACTION)
    replay_images.append(obs["agentview_image"][::-1])

In [16]:
import imageio
imageio.mimwrite('test_dp_output.mp4', replay_images, fps=20)
replay_images = []